In [1]:
# 1. Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# 2. Install dependencies
%%capture
!pip install unsloth
!pip install --upgrade trl peft accelerate bitsandbytes datasets

In [4]:
# 3. Check data files
import os
assert os.path.exists('/content/train.jsonl'), 'train.jsonl not found'
assert os.path.exists('/content/val.jsonl'), 'val.jsonl not found'
print('Data files found!')

Data files found!


In [5]:
# 4. Load model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,
)
print('Model loaded')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded


In [6]:
# 5. Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.3 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [8]:
# 6. Load dataset
from datasets import load_dataset
dataset = load_dataset('json', data_files={'train': 'data/train.jsonl', 'validation': 'data/val.jsonl'})
print('Train:', len(dataset['train']), '| Val:', len(dataset['validation']))

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train: 121 | Val: 14


In [11]:
# 7. Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=2048,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        eval_steps=100,
        save_steps=100,
        output_dir='/content/checkpoints',
        optim='adamw_8bit',
        seed=42,
        report_to='none',
    ),
)
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/121 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/14 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 121 | Num Epochs = 3 | Total steps = 48
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
10,3.427541
20,2.709553
30,1.706324
40,1.230790


TrainOutput(global_step=48, training_loss=2.0959043304125466, metrics={'train_runtime': 161.2343, 'train_samples_per_second': 2.251, 'train_steps_per_second': 0.298, 'total_flos': 1256809341413376.0, 'train_loss': 2.0959043304125466})

In [12]:
# 8. Save and download adapter
import shutil
model.save_pretrained('nepallawft-lora')
tokenizer.save_pretrained('nepallawft-lora')
shutil.make_archive('nepallawft-lora', 'zip', 'nepallawft-lora')
files.download('nepallawft-lora.zip')
print('Done')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done


In [13]:
# 9. Quick test
FastLanguageModel.for_inference(model)

SYSTEM = 'You are CivicLens, a legal assistant for Nepal\'s laws and governance documents.'

def ask(question):
    prompt = (f'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{SYSTEM}<|eot_id|>'
              f'<|start_header_id|>user<|end_header_id|>\n{question}<|eot_id|>'
              f'<|start_header_id|>assistant<|end_header_id|>\n')
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split('assistant')[-1].strip()

print(ask('What does the Nepal constitution say about fundamental rights?'))
print('---')
print(ask('भ्रष्टाचारको सजाय के हो?'))

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

The Nepal Constitution (2017) defines fundamental rights as the basic rights that every individual in Nepal is entitled to. These rights are: (1) Right to life, (2) Right to physical integrity, (3) Right to dignity, (4) Right to freedom from exploitation, (5) Right to freedom from discrimination, (6) Right to equality, (7) Right to freedom of expression, (8) Right to freedom of association, (9) Right to freedom of movement, (10) Right to participate in the decision-making process, and (11) Right to work, among others.
---
सजाय हुने के हुन्छन्: 6.6.सवारीको व्यय, सवारीको व्यवस्था र सवारीको सेवा द्वारा भ्रष्टाचार के हुन्छन्? 6.6.1. सवारीको व्यय, सवारीको व्यवस्था र सवारीको सेवा द्वारा भ्रष्टाचार के हुन्छन्? 6.6.1. सवारीको व्यय, सवारीको व्यवस्था र सवारीको सेवा द्वारा भ्रष्टाचार के हुन्छन्? 6.6.1. सवारीको व्यय, सवारीको व्यवस्था र सवारीको सेवा द्वारा भ्रष्टाचार के हुन्छन्? 6.6.1. सवारीको व्यय, सवारीको व्यवस्था र सवारीको सेवा द्वारा भ्रष्टाचार के हुन्छन
